<div align="center">

<a href="https://colab.research.google.com/github/utkukose/cyber-physical-systems-NB-lecture/blob/main/exams/final/FIN1_heated_hall_twin.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"></a>

# Final option 1: A heated building with its digital twin

**Cyber-Physical Systems (BIL-435 Siber Fiziksel Sistemler)**<br>Final project: A networked cyber-physical system with its digital twin. Starter Colab notebook

[Project page](https://github.com/utkukose/cyber-physical-systems-NB-lecture/tree/main/exams/final) | [Course repository](https://github.com/utkukose/cyber-physical-systems-NB-lecture)

**Prof. Dr. Utku Kose**

Full Professor, Department of Computer Engineering, Süleyman Demirel University, Isparta, Türkiye  
Founding Director, AI Application and Research Center (YAZEM), Süleyman Demirel University  
Head of the Computer Science Division, Department of Computer Engineering, Süleyman Demirel University  
Additional affiliations: University of North Dakota (USA), Universidad Panamericana (Mexico City, Mexico), Vel Tech University (Chennai, India)  
IEEE Senior Member, ACM Professional Member

[ORCID 0000-0002-9652-6415](https://orcid.org/0000-0002-9652-6415) | [utkukose.com](https://www.utkukose.com) | [github.com/utkukose](https://github.com/utkukose)

[utkukose@sdu.edu.tr](mailto:utkukose@sdu.edu.tr) | [utku.kose@und.edu](mailto:utku.kose@und.edu) | [ukose@up.edu.mx](mailto:ukose@up.edu.mx) | [utkukose@gmail.com](mailto:utkukose@gmail.com)

<sub>Content licensed under CC BY 4.0, code under MIT. This course is updated in line with current developments in the field. First version: November 2026.</sub>

</div>

## How to use this starter Colab notebook

Save a copy in Drive first, with File, Save a copy in Drive. Every code cell runs as it is with an example case: A sports hall with a heat pump of 60 kW and wireless sensors. Replace the values marked `# TODO` with the data of your own system. Then run all cells again from the top, with Runtime, Run all, and use the printed values and figures in the report. Each section names the week and the lecture section that explain its method. The last section collects the numbers for the results table of the report. Keep your own data in the cells, so that the Colab notebook documents how every number of the report was obtained.

## 0. Setup

The cell loads the libraries and creates the dictionary `RESULTS`, which collects the numbers of the report.

In [ ]:
import math

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import least_squares, minimize_scalar

RESULTS = {}            # the numbers for the results table of the report
print("Setup complete.")

## 1. The communication of the hall

*Lecture: [Week 8, Publish and subscribe](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-08/lecture.html#mqtt) and [Week 8, Latency, queues and repetition](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-08/lecture.html#latency)*

The sensors of the hall publish their readings to a Message Queuing Telemetry Transport (MQTT) broker, and the twin,
a dashboard and a logger subscribe to them. A group of $n_i$ devices that publish every $T_i$ seconds sends
$\lambda = \sum_i n_i / T_i$ messages per second. With the quality of service (QoS) level of MQTT, each message costs
$k = 1, 2$ or $4$ packets for the publisher and as many for each of the $S$ subscribers, so the broker handles
$\lambda_p = \lambda\,(k + S\,k)$ packets per second. With the service rate $\mu$, the broker is a queue with

$$\rho = \frac{\lambda_p}{\mu}, \qquad W = \frac{1}{\mu - \lambda_p}, \qquad L = \lambda_p\,W ,$$

the utilisation $\rho$, the mean time $W$ of a packet in the broker and the mean number $L$ of packets in it, by Little's
law. A publisher repeats a message until it is acknowledged. With the probability $p$ that one transmission is lost,
all $k_a$ attempts fail with the probability $P_{miss} = p^{k_a}$. The cell checks the load of the broker, the plan of the
cloud provider and the readings lost per day.

In [ ]:
# TODO: the publishers and the broker of your system
PUBLISHERS = pd.DataFrame([
    # topic, devices, size of a message (bytes), period (s)
    ("school/hall/air/+/temperature", 4, 60, 60),
    ("school/hall/air/+/co2", 4, 60, 60),
    ("school/hall/heatpump/heat", 1, 80, 10),
    ("school/hall/heatpump/status", 1, 120, 60),
    ("school/hall/door/+/state", 6, 40, 30),
    ("school/weather/temperature", 1, 60, 300),
], columns=["topic", "devices", "bytes", "period (s)"])
SUBSCRIBERS = 3              # the twin, the dashboard and the logger subscribe to every topic
K_PACKETS = 2                # packets per message and receiver: 1, 2 or 4 for QoS 0, 1 or 2
MU = 50.0                    # packets per second that the broker serves
P_LOSS = 0.02                # probability that one transmission is lost
ATTEMPTS = 3                 # attempts of a publisher before a reading is lost
PLAN_PER_MONTH = 2_000_000   # messages per month in the plan of the cloud broker

PUBLISHERS["messages per s"] = PUBLISHERS["devices"] / PUBLISHERS["period (s)"]
PUBLISHERS["bytes per day"] = PUBLISHERS["messages per s"] * PUBLISHERS["bytes"] * 86400
lam = PUBLISHERS["messages per s"].sum()
lam_p = lam * (K_PACKETS + SUBSCRIBERS * K_PACKETS)
rho, W = lam_p / MU, 1 / (MU - lam_p)
month = lam * 86400 * 30
p_miss = P_LOSS ** ATTEMPTS
print(PUBLISHERS.round(4).to_string(index=False))
print(f"lambda = {lam:.3f} messages/s, {lam_p:.2f} packets/s, rho = {rho:.3f}, W = {1000 * W:.1f} ms, "
      f"L = {lam_p * W:.3f} packets")
print(f"Messages per month: {month:,.0f} of {PLAN_PER_MONTH:,} in the plan ({100 * month / PLAN_PER_MONTH:.0f} %)")
print(f"P_miss = {p_miss:.1e}: {lam * 86400 * p_miss:.2f} readings lost per day")
RESULTS["messages per second"] = round(lam, 3)
RESULTS["utilisation of the broker"] = round(rho, 3)
RESULTS["mean time in the broker (ms)"] = round(1000 * W, 1)
RESULTS["share of the cloud plan (%)"] = round(100 * month / PLAN_PER_MONTH, 1)
RESULTS["readings lost per day"] = round(lam * 86400 * p_miss, 2)

## 2. Calibrating the twin

*Lecture: [Week 12, What a digital twin is](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-12/lecture.html#digital-twin) and [Week 12, Calibration: Fitting the twin to the library](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-12/lecture.html#calibration)*

The twin is the heat model of Week 12. Every quarter of an hour, it advances the temperature $T$ of the hall with

$$T_{k+1} = T_k + \frac{\Delta t}{C}\left(\frac{T_{out,k} - T_k}{R} + P_k + g\,G_k + q\,o_k\right),$$

where $P$ is the heat of the heat pump in kW, $G$ the solar irradiance in kW/m² and $o$ the number of people. The
parameters $C$ and $R$ are the capacity and the resistance of Section 1 of the midterm. The solar aperture $g$ is given
in m², and $q$ is the heat of one person in kW. The function `least_squares` finds the parameters that minimise the
squared difference between the simulated and the measured temperatures of the first week. The days after it validate
the twin: Its root mean square error (RMSE) there shows how well it predicts temperatures that it has not seen. The
example log is simulated, and from the morning of day 10 a roof hatch stays open.

In [ ]:
# TODO: replace the simulated log with the log of your system: one row per 15 minutes
DT_H = 0.25                  # step of the log and of the twin (h)
P_HP = 60.0                  # heat of the heat pump at full power (kW)
TRAIN_DAYS, VALID_DAYS = 7, 2

def hall_history(days=14, seed=21, open_from=9 * 24 + 8):
    """A simulated log of the hall under its thermostat; the true model changes when the hatch opens."""
    rng = np.random.default_rng(seed)
    per_day = int(24 / DT_H)
    t = np.arange(days * per_day) * DT_H                         # hours since the start
    hour, day = t % 24, (t // 24).astype(int)
    t_out = 3 + 4 * np.sin(2 * np.pi * (hour - 9) / 24) + np.repeat(rng.normal(0, 2.0, days), per_day)
    G = 0.45 * np.clip(np.sin(np.pi * (hour - 8) / 9), 0, None) * np.repeat(rng.uniform(0.2, 1.0, days), per_day)
    weekday = day % 7 < 5
    people = np.where(weekday & (hour >= 9) & (hour < 15), 25, 0) + np.where(weekday & (hour >= 16) & (hour < 22), 40, 0)
    temp, heat, rows = 16.0, False, []
    for k in range(t.size):
        r = 18.0 if 6 <= hour[k] < 22 else 14.0
        y = temp + rng.normal(0, 0.1)                            # the reading of the hall sensor
        if heat and y > r + 0.5:
            heat = False
        elif not heat and y < r - 0.5:
            heat = True
        P = P_HP if heat else 0.0
        rows.append((t[k], t_out[k], G[k], people[k], P, y, temp))
        R = 0.40 if t[k] >= open_from else 0.55
        temp += DT_H / 30.0 * ((t_out[k] - temp) / R + P + 12.0 * G[k] + 0.15 * people[k]) + rng.normal(0, 0.03)
    return pd.DataFrame(rows, columns=["hour", "t_out", "G", "people", "P", "t_in", "t_true"])

LOG = hall_history()
n_train, n_valid = int(TRAIN_DAYS * 24 / DT_H), int(VALID_DAYS * 24 / DT_H)
TRAIN, VALID = LOG.iloc[:n_train], LOG.iloc[n_train:n_train + n_valid]

def simulate(d, C, R, g, q, gain=0.0):
    """The twin over the log d. With a gain, it corrects its state with each reading. Returns the twin after the
    corrections and its predictions before them."""
    temp, twin, pred = d.t_in.iloc[0], [], []
    for row in d.itertuples():
        pred.append(temp)
        temp += gain * (row.t_in - temp)
        twin.append(temp)
        temp += DT_H / C * ((row.t_out - temp) / R + row.P + g * row.G + q * row.people)
    return np.array(twin), np.array(pred)

def rmse(a, b):
    return float(np.sqrt(np.mean((np.asarray(a) - np.asarray(b)) ** 2)))

fit = least_squares(lambda p: simulate(TRAIN, *p)[0] - TRAIN.t_in.values, x0=[20.0, 0.5, 5.0, 0.1],
                    bounds=([1.0, 0.05, 0.0, 0.0], [200.0, 5.0, 100.0, 1.0]))
FIT = dict(zip(["C", "R", "g", "q"], fit.x))
s2 = 2 * fit.cost / (len(TRAIN) - 4)                             # variance of the residuals
cov = s2 * np.linalg.inv(fit.jac.T @ fit.jac)                    # covariance of the parameters
SE = dict(zip(["C", "R", "g", "q"], np.sqrt(np.diag(cov))))
for k in FIT:
    print(f"{k} = {FIT[k]:.3f} (standard error {SE[k]:.3f})")
print(f"RMSE: training {rmse(simulate(TRAIN, **FIT)[0], TRAIN.t_in):.3f} K, "
      f"validation {rmse(simulate(VALID, **FIT)[0], VALID.t_in):.3f} K")
for k in FIT:
    RESULTS[f"twin: {k}"] = round(FIT[k], 3)
RESULTS["twin: RMSE of the validation (K)"] = round(rmse(simulate(VALID, **FIT)[0], VALID.t_in), 3)

fig, ax = plt.subplots(figsize=(8, 3.2))
ax.plot(VALID.hour / 24, VALID.t_in, lw=0.8, color="#9AA5AE", label="measured")
ax.plot(VALID.hour / 24, simulate(VALID, **FIT)[0], label="twin, open loop")
ax.set_xlabel("Day")
ax.set_ylabel("Temperature (°C)")
ax.legend(fontsize=8)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 3. Keeping the twin in step

*Lecture: [Week 12, Synchronisation: Keeping the twin in step](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-12/lecture.html#synchronisation) and [Week 12, Drift and recalibration](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-12/lecture.html#recalibration)*

A synchronised twin corrects its state with each reading, $T_k \leftarrow T_k + \kappa\,(y_k - T_k)$, with a gain
$\kappa$ between 0 and 1. Its innovations, the readings minus the predictions, stay small while the model is right. A
change of the hall, such as an open hatch, shifts them. The cumulative sum (CUSUM) of Week 10 watches the standardised
innovations $z_k$:

$$S^+_k = \max(0,\ S^+_{k-1} + z_k - \kappa_c), \qquad S^-_k = \min(0,\ S^-_{k-1} + z_k + \kappa_c),$$

and alarms when $S^+_k > h$ or $S^-_k < -h$. After an alarm, the resistance $R$ is fitted again over the next 24 hours,
with $C$, $g$ and $q$ kept. The open hatch then adds the conductance $1/R_{new} - 1/R$ in kW/K.

In [ ]:
# TODO: the synchronisation and the drift detection of your twin
GAIN = 0.3                   # gain of the synchronisation
KAPPA_C, H_C = 0.5, 5.0      # allowance and threshold of the CUSUM
OPEN_FROM = 9 * 24 + 8       # hour at which the hatch was opened in the example log (unknown to the twin)

def cusum(z, kappa=KAPPA_C, h=H_C):
    """Positions of the alarms of the CUSUM; it starts again at zero after each one."""
    sp = sm = 0.0
    alarms = []
    for k, v in enumerate(z):
        sp, sm = max(0.0, sp + v - kappa), min(0.0, sm + v + kappa)
        if sp > h or sm < -h:
            alarms.append(k)
            sp = sm = 0.0
    return alarms

sigma = float(np.std(VALID.t_in.values[1:] - simulate(VALID, **FIT, gain=GAIN)[1][1:]))
WATCH = LOG.iloc[n_train:]
twin, pred = simulate(WATCH, **FIT, gain=GAIN)
alarms = WATCH.hour.values[cusum((WATCH.t_in.values - pred) / sigma)]
false = int((alarms < OPEN_FROM).sum())
first = next((a for a in alarms if a >= OPEN_FROM), None)
print(f"Spread of the innovations {sigma:.3f} K; false alarms before the opening: {false}")
print(f"First alarm {first - OPEN_FROM:.2f} h after the opening" if first is not None else "No alarm after the opening")

if first is not None:
    window = LOG[(LOG.hour > first) & (LOG.hour <= first + 24)]
    err = lambda r: rmse(simulate(window, FIT["C"], r, FIT["g"], FIT["q"])[0], window.t_in)
    R_new = minimize_scalar(err, bounds=(0.05, 2.0), method="bounded").x
    print(f"R recalibrated over the 24 h after the alarm: {R_new:.3f} K/kW (calibration {FIT['R']:.3f} K/kW); "
          f"the hatch adds {1 / R_new - 1 / FIT['R']:.2f} kW/K")
    RESULTS["CUSUM: hours from the opening to the alarm"] = round(first - OPEN_FROM, 2)
    RESULTS["recalibrated R (K/kW)"] = round(R_new, 3)
RESULTS["CUSUM: false alarms"] = false

fig, ax = plt.subplots(figsize=(8, 3.2))
innov = WATCH.t_in.values - pred
ax.plot(WATCH.hour / 24, innov, lw=0.8, label="innovation")
ax.axvline(OPEN_FROM / 24, color="#B23B3B", ls="--", lw=1, label="hatch opened")
ax.plot(alarms / 24, np.full(len(alarms), innov.max() + 0.05), "|", ms=10, color="#C98510", label="alarms of the CUSUM")
ax.set_xlabel("Day")
ax.set_ylabel("Reading minus prediction (K)")
ax.legend(fontsize=8)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Asking the twin: When to start on Monday

*Lecture: [Week 13, What-if analysis with the twin](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-13/lecture.html#what-if) and [Week 13, Uncertainty: Many possible weeks](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-13/lecture.html#monte-carlo)*

On Monday, the first lesson starts at 8:00, and the hall must reach 17 °C by then. The heat pump keeps 14 °C over the
weekend and runs at full power from a start time $t_s$. A later start saves energy, but a cold morning or a wrong
parameter can leave the hall too cold. The Monte Carlo method answers the question with many possible mornings. Each
draw takes an outdoor temperature from the forecast with its error and parameters from the calibration with their
standard errors. With $n$ draws, the share $\hat{p}$ of cold mornings has the standard error

$$\sigma_{\hat{p}} = \sqrt{\hat{p}\,(1 - \hat{p})/n} .$$

The cell runs the twin from midnight for several start times, with the hatch closed again. It chooses the latest start
whose probability of a cold hall stays below the target.

In [ ]:
# TODO: the question to your twin
T_NEED, T_NIGHT = 17.0, 14.0           # needed at 8:00 and kept over the night (°C)
FORECAST = 0.0                          # forecast of the mean outdoor temperature on Monday morning (°C)
FORECAST_SD = 2.0                       # error of the forecast (K)
STARTS = [3.0, 4.0, 4.5, 5.0, 5.5, 6.0, 6.5]
TARGET = 0.05                           # largest acceptable probability of a cold hall at 8:00
DRAWS = 1000

def morning(start, t_out, C, R, g, q):
    """Temperature at 8:00 and energy of the heat pump (kWh) for a start at the hour start."""
    temp, energy = T_NIGHT, 0.0
    for h in np.arange(0, 8, DT_H):
        if h >= start:
            P = P_HP
        else:
            P = min(P_HP, max(0.0, (T_NIGHT - t_out) / R))           # holds the night temperature
        temp += DT_H / C * ((t_out - temp) / R + P)
        energy += P * DT_H
    return temp, energy

rng = np.random.default_rng(13)
params = {k: rng.normal(FIT[k], SE[k], DRAWS) for k in ("C", "R")}
t_outs = rng.normal(FORECAST, FORECAST_SD, DRAWS)
rows = []
for s in STARTS:
    res = np.array([morning(s, t_outs[i], params["C"][i], params["R"][i], 0, 0) for i in range(DRAWS)])
    p = float(np.mean(res[:, 0] < T_NEED))
    rows.append({"start (h)": s, "median at 8:00 (°C)": round(float(np.median(res[:, 0])), 2),
                 "P(cold)": round(p, 3), "standard error": round(math.sqrt(p * (1 - p) / DRAWS), 3),
                 "mean energy (kWh)": round(float(res[:, 1].mean()), 1)})
table = pd.DataFrame(rows)
print(table.to_string(index=False))
ok = table[table["P(cold)"] <= TARGET]
if len(ok):
    best = ok.iloc[-1]
    print(f"Latest start with P(cold) <= {TARGET}: {best['start (h)']} h, energy {best['mean energy (kWh)']} kWh")
    RESULTS["latest safe start (h)"] = float(best["start (h)"])
    RESULTS["P(cold) at that start"] = float(best["P(cold)"])
    RESULTS["energy of the morning (kWh)"] = float(best["mean energy (kWh)"])

## 5. The safety argument: Frost in the empty hall

*Lecture: [Week 14, Fault trees](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-14/lecture.html#fault-trees) and [Week 14, Hidden faults and common causes](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-14/lecture.html#hidden-faults) and [Week 14, Runtime monitoring and the simplex architecture](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-14/lecture.html#runtime-monitoring)*

During a holiday, the hall is empty, and its showers freeze if the temperature falls below 5 °C. The fault tree has the
top event *frost*. Heat is lost when the heat pump fails or when the sensor of the hall sticks at a warm value, so that
the controller stops heating. A frost thermostat should then start a backup heater, and the caretaker should get an
alarm. Frost needs the loss of heat, a failed backup and a missed alarm. With independent basic events, an AND gate
multiplies the probabilities, and an OR gate gives $1 - \prod_i (1 - p_i)$. The frost thermostat has hidden faults with
the rate $\lambda$, found only by proof tests every $\tau$ years, so it fails on demand with $q = \lambda\tau/2$.

A runtime monitor compares the sensor with the twin. When the reading stays far from the prediction, it alarms and
switches the controller to the twin. The cell first simulates a sensor that sticks and measures how fast the monitor
notices it. It then computes the probability of frost with and without the monitor. The Fussell-Vesely importance of a
basic event is the share of the top probability in the cut sets that contain it.

In [ ]:
# TODO: the probabilities of your system for one winter
P_HP_FAIL = 0.05             # the heat pump stops during a holiday and is not repaired in time
P_STUCK = 0.02               # the sensor of the hall sticks at a warm value
LAM_FT, TAU_FT = 0.1, 1.0    # hidden faults of the frost thermostat per year, years between proof tests
P_BACKUP = 0.01              # the backup heater fails when it is started
P_ALARM = 0.05               # the alarm does not reach the caretaker in time
THRESH, COUNT = 1.0, 8       # the monitor alarms after COUNT readings in a row that differ by more than THRESH (K)
STUCK_AT = 12.0              # value of the stuck sensor (°C)

def holiday(monitor, hours=72, seed=8):
    """A holiday at 10 °C; the sensor sticks at hour 12. The hall follows the true model of the example log, and the
    monitor compares the reading with the twin, which runs with the calibrated parameters."""
    rng = np.random.default_rng(seed)
    temp, twin_t, run_len, detect, rows = 10.0, 10.0, 0, None, []
    for h in np.arange(0, hours, DT_H):
        t_o = -4 + 3 * np.sin(2 * np.pi * (h - 9) / 24)
        y = STUCK_AT if h >= 12 else temp + rng.normal(0, 0.1)
        run_len = run_len + 1 if abs(y - twin_t) > THRESH else 0
        if monitor and detect is None and run_len >= COUNT:
            detect = h
        value = twin_t if detect is not None else y                 # after the alarm, the controller uses the twin
        P = min(P_HP, max(0.0, (10.0 - t_o) / FIT["R"] + 5 * (10.0 - value)))
        rows.append((h, temp, y, twin_t, P))
        temp += DT_H / 30.0 * ((t_o - temp) / 0.55 + P) + rng.normal(0, 0.03)
        twin_t += DT_H / FIT["C"] * ((t_o - twin_t) / FIT["R"] + P)
    return pd.DataFrame(rows, columns=["hour", "true", "reading", "twin", "P"]), detect

with_m, detect = holiday(True)
without_m, _ = holiday(False)
print(f"Sensor stuck at hour 12; the monitor alarms at hour {detect} ({detect - 12:.2f} h later)")
print(f"Lowest true temperature: {with_m.true.min():.2f} °C with the monitor, "
      f"{without_m.true.min():.2f} °C without it")

def frost(p_stuck):
    q_ft = LAM_FT * TAU_FT / 2
    loss = 1 - (1 - P_HP_FAIL) * (1 - p_stuck)
    backup = 1 - (1 - q_ft) * (1 - P_BACKUP)
    return loss * backup * P_ALARM, q_ft

COVERAGE = 0.95              # share of stuck sensors that the monitor detects in time
p0, q_ft = frost(P_STUCK)
p1, _ = frost(P_STUCK * (1 - COVERAGE))
print(f"q of the frost thermostat = {q_ft:.3f}")
print(f"P(frost) per winter: without the monitor {p0:.2e}, with it {p1:.2e}")
cuts = {"heat pump, thermostat, alarm": P_HP_FAIL * q_ft * P_ALARM,
        "heat pump, backup, alarm": P_HP_FAIL * P_BACKUP * P_ALARM,
        "sensor, thermostat, alarm": P_STUCK * q_ft * P_ALARM,
        "sensor, backup, alarm": P_STUCK * P_BACKUP * P_ALARM}
total = sum(cuts.values())
for event in ["heat pump", "sensor", "thermostat", "backup", "alarm"]:
    print(f"Fussell-Vesely importance of the {event}: {sum(v for c, v in cuts.items() if event in c) / total:.3f}")
RESULTS["monitor: hours to detect a stuck sensor"] = round(detect - 12, 2)
RESULTS["lowest temperature without the monitor (°C)"] = round(float(without_m.true.min()), 2)
RESULTS["P(frost) without the monitor"] = float(f"{p0:.3g}")
RESULTS["P(frost) with the monitor"] = float(f"{p1:.3g}")

fig, ax = plt.subplots(figsize=(8, 3.2))
ax.plot(with_m.hour, with_m.true, label="true, with the monitor")
ax.plot(without_m.hour, without_m.true, label="true, without the monitor")
ax.plot(with_m.hour, with_m.reading, lw=0.8, color="#9AA5AE", label="reading of the sensor")
ax.axvline(detect, color="#C98510", lw=1, label="alarm of the monitor")
ax.axhline(5.0, color="#B23B3B", ls="--", lw=1, label="frost limit")
ax.set_xlabel("Hour of the holiday")
ax.set_ylabel("Temperature (°C)")
ax.legend(fontsize=8)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

Write the safety argument of the report as a claim, supported by evidence. The evidence here is the simulation of the stuck sensor, the fault tree with its numbers and the assumptions behind them. Name the assumption that you trust least.

## 6. Results for the report

The cell collects the numbers that the sections above stored in `RESULTS`. Copy them into the results table of the report, and explain each one in the text. A value that changes the design deserves a figure.

In [ ]:
import pandas as pd

print(pd.Series(RESULTS, name="value", dtype=object).to_string())

---

<div align="center">

**Prof. Dr. Utku Kose**

Full Professor, Department of Computer Engineering, Süleyman Demirel University, Isparta, Türkiye  
Founding Director, AI Application and Research Center (YAZEM), Süleyman Demirel University  
Head of the Computer Science Division, Department of Computer Engineering, Süleyman Demirel University  
Additional affiliations: University of North Dakota (USA), Universidad Panamericana (Mexico City, Mexico), Vel Tech University (Chennai, India)  
IEEE Senior Member, ACM Professional Member

[ORCID 0000-0002-9652-6415](https://orcid.org/0000-0002-9652-6415) | [utkukose.com](https://www.utkukose.com) | [github.com/utkukose](https://github.com/utkukose)

[utkukose@sdu.edu.tr](mailto:utkukose@sdu.edu.tr) | [utku.kose@und.edu](mailto:utku.kose@und.edu) | [ukose@up.edu.mx](mailto:ukose@up.edu.mx) | [utkukose@gmail.com](mailto:utkukose@gmail.com)

This course is updated in line with current developments in the field. First version: November 2026.

</div>